# Eclectic FIRE Analysis and Python Modeling
## A Comprehensive Study of Financial Independence, Retire Early in the Kenyan Context

This notebook contains a complete financial analysis modeling the path to **Financial Independence, Retire Early (FIRE)**. It derives the mathematical pillars of the FIRE framework, implements computational functions in Python, and analyzes a specific case study of car ownership (including fuel and financing payments) to illustrate the opportunity cost of lifestyle expenses on a retirement timeline.

### Pillar 1: The Core Math of FIRE

#### 1. Savings Rate ($SR$)
The savings rate is the single most important variable in the wealth-accumulation phase. It is defined as:
$$SR = \frac{\text{Income} - \text{Expenses}}{\text{Income}} = \frac{\text{Savings}}{\text{Income}}$$

#### 2. The Safe Withdrawal Rate ($SWR$) and the FIRE Number
The **FIRE Number** is the total portfolio size needed to sustain your expenses in retirement. It is based on the Safe Withdrawal Rate (SWR), which represents the percentage of your portfolio you can withdraw annually without running out of money.
$$\text{FIRE Number} = \frac{\text{Annual Retirement Expenses}}{\text{SWR}}$$
For example, using the traditional $4\%$ rule (derived from the Trinity Study):
$$\text{FIRE Number} = \text{Annual Expenses} \times 25$$
For a more conservative $3.5\%$ SWR (recommended for early retirement horizons of 40+ years in volatile or high-inflation regions):
$$\text{FIRE Number} = \text{Annual Expenses} \times 28.57$$

In [ ]:
def calculate_savings_rate(income, expenses):
    """Calculates the savings rate as a percentage of income."""
    if income <= 0:
        raise ValueError("Income must be positive.")
    savings = income - expenses
    return (savings / income) * 100

def calculate_fire_number(annual_expenses, withdrawal_rate=0.035):
    """Calculates the target portfolio value (FIRE number) based on withdrawal rate."""
    if withdrawal_rate <= 0:
        raise ValueError("Withdrawal rate must be positive.")
    return annual_expenses / withdrawal_rate

# Example usage representing the user's initial state:
income = 15480
savings = 4644
expenses = income - savings
sr = calculate_savings_rate(income, expenses)
print(f"Current Monthly Income: KES {income:,}")
print(f"Current Monthly Savings: KES {savings:,} (Savings Rate: {sr:.1f}%)")

# Setting an aspirational retirement spending budget of KES 100,000 per month (in today's money)
target_annual_expenses_today = 100000 * 12 
fire_num_today = calculate_fire_number(target_annual_expenses_today, 0.035)
print(f"FIRE Target Number (Today's Value, 3.5% SWR): KES {fire_num_today:,.2f}")

### Pillar 2: Compound Interest, Inflation, and Portfolio Growth

#### 1. Future Value of a Portfolio
With monthly contributions compounding, the future value ($FV$) of a series of equal monthly investments ($PMT$) is modeled using the Future Value of an Ordinary Annuity:
$$FV = PMT \times \frac{(1 + r_m)^n - 1}{r_m} + PV \times (1 + r_m)^n$$
Where:
* $PV$ = Present Value (initial savings)
* $PMT$ = Monthly investment amount
* $r_m$ = Monthly interest rate ($r_{\text{annual}} / 12$)
* $n$ = Number of compounding periods (months, $t \times 12$)

#### 2. Nominal vs. Real Returns (Adjusting for Inflation)
Inflation erodes purchasing power. To model the **real** growth of a portfolio, we adjust nominal returns using the Fisher Equation:
$$1 + r_{\text{nominal}} = (1 + r_{\text{real}})(1 + i)$$
$$r_{\text{real}} = \frac{1 + r_{\text{nominal}}}{1 + i} - 1$$
Where:
* $r_{\text{nominal}}$ = Nominal expected return (e.g., $12\%$)
* $i$ = Expected annual inflation rate (e.g., $6\%$)

In [ ]:
def project_portfolio(current_savings, monthly_investment, annual_return, inflation, years):
    """Projects nominal and real portfolio growth over a given number of years.
    Assumes contributions are made monthly and interest compounds monthly.
    """
    months = years * 12
    monthly_nominal_rate = annual_return / 12
    
    # Nominal projection calculation
    nominal_balance = current_savings
    nominal_history = []
    for month in range(1, months + 1):
        nominal_balance = nominal_balance * (1 + monthly_nominal_rate) + monthly_investment
        nominal_history.append(nominal_balance)
        
    # Real projection calculation (deflating balance back to today's purchasing power)
    real_history = []
    for month in range(1, months + 1):
        yr = month / 12
        real_bal = nominal_history[month - 1] / ((1 + inflation) ** yr)
        real_history.append(real_bal)
        
    return nominal_history, real_history

# Example: Projecting 27 years of growth (Retirement Horizon from Age 23 to 50)
current_savings = 173000
monthly_investment = 4644
annual_return = 0.12
inflation = 0.06
years = 27

nominal, real = project_portfolio(current_savings, monthly_investment, annual_return, inflation, years)
print(f"Projected Nominal Portfolio after 27 years: KES {nominal[-1]:,.2f}")
print(f"Projected Real Portfolio (Today's Purchasing Power): KES {real[-1]:,.2f}")

### Pillar 3: Case Study - The Impact of Car Financing & Fuel Expenses

Many professionals purchase a vehicle early in their careers without realizing the full opportunity cost of that decision. Here we model the purchase of a car with a financing payment of **KES 7,000 / month** and recurring operating fuel expenditures of **KES 7,000 / month**, totaling **KES 14,000 / month** in car-related cash outflow.

#### Budgeting Impact on KES 59,000/month salary (Phase 2):
Under a 50/30/20 budget framework:
* **Essentials (50%)**: KES 29,500
* **Investments (30%)**: KES 17,700 (Savings rate target)
* **Entertainment (20%)**: KES 11,800

When introducing the vehicle:
* The KES 7,000 fuel cost is a recurring operating cost (Essential).
* The KES 7,000 car financing is a fixed liability payment.
* If the user maintains Essentials (housing, food, utilities) and Entertainment constant, the entire KES 14,000 car expense eats directly into their investments. The monthly savings drop from **KES 17,700** to **KES 3,700** (a $79\%$ drop in savings rate!).

In [ ]:
def calculate_opportunity_cost(monthly_cost, annual_return, years):
    """Calculates the compound opportunity cost of a recurring monthly expense over time."""
    months = years * 12
    monthly_rate = annual_return / 12
    # Future value of an annuity formula
    fv = monthly_cost * (((1 + monthly_rate) ** months - 1) / monthly_rate)
    return fv

years_to_retire = 27
nominal_return = 0.12
inflation_rate = 0.06
initial_savings = 173000

fuel_expense = 7000
car_purchase = 7000
total_car_expense = fuel_expense + car_purchase

# Calculate Opportunity Costs (Future Value at Age 50)
opp_cost_fuel = calculate_opportunity_cost(fuel_expense, nominal_return, years_to_retire)
opp_cost_car = calculate_opportunity_cost(car_purchase, nominal_return, years_to_retire)
opp_cost_total = opp_cost_fuel + opp_cost_car

print("=== COMPOND OPPORTUNITY COST ANALYSIS AT AGE 50 ===")
print(f"Monthly Fuel Cost (KES 7,000) Opportunity Cost: KES {opp_cost_fuel:,.2f}")
print(f"Monthly Car Loan (KES 7,000) Opportunity Cost: KES {opp_cost_car:,.2f}")
print(f"Total Opportunity Cost of Vehicle: KES {opp_cost_total:,.2f}\n")

# Compare Trajectory Scenarios
baseline_monthly = 17700              # Baseline 30% savings
with_fuel_monthly = baseline_monthly - fuel_expense       # KES 10,700 savings
with_car_monthly = baseline_monthly - total_car_expense   # KES 3,700 savings

# Portfolios at age 50
nom_base, _ = project_portfolio(initial_savings, baseline_monthly, nominal_return, inflation_rate, years_to_retire)
nom_fuel, _ = project_portfolio(initial_savings, with_fuel_monthly, nominal_return, inflation_rate, years_to_retire)
nom_car, _ = project_portfolio(initial_savings, with_car_monthly, nominal_return, inflation_rate, years_to_retire)

print("=== NOMINAL PORTFOLIO COMPARISON AT AGE 50 ===")
print(f"Scenario A: Baseline (No Car): KES {nom_base[-1]:,.2f}")
print(f"Scenario B: Own Car Outright (Fuel KES 7K/mo): KES {nom_fuel[-1]:,.2f}")
print(f"Scenario C: Finance + Fuel (Total KES 14K/mo): KES {nom_car[-1]:,.2f}\n")

# Calculate Timeline Delay (how long to reach baseline target of KES 39.47M)
target_val = nom_base[-1]

def years_to_target(current, monthly, target, rate):
    monthly_rate = rate / 12
    months = 0
    balance = current
    while balance < target and months < 1200:
        balance = balance * (1 + monthly_rate) + monthly
        months += 1
    return months / 12

t_base = years_to_retire
t_fuel = years_to_target(initial_savings, with_fuel_monthly, target_val, nominal_return)
t_car = years_to_target(initial_savings, with_car_monthly, target_val, nominal_return)

print("=== RETIREMENT TIMELINE DELAY ANALYSIS ===")
print(f"Scenario A (Baseline): Reaches target in {t_base:.1f} years (Retire at 50)")
print(f"Scenario B (Fuel Only): Takes {t_fuel:.1f} years (Delay: +{t_fuel - t_base:.1f} years, Retire at {50 + (t_fuel - t_base):.1f})")
print(f"Scenario C (Finance + Fuel): Takes {t_car:.1f} years (Delay: +{t_car - t_base:.1f} years, Retire at {50 + (t_car - t_base):.1f})")

### Pillar 4: Risk Management & Strategic Takeaways

#### Key Takeaways:
1. **Opportunity Cost Leverage**: Recurring small expenses early in your career carry an immense compounding weight. A vehicle costing KES 14,000/month over 27 years reduces your retirement portfolio by **KES 28.4 Million** due to lost compounding yields (at $12\%$ p.a.).
2. **Retirement Timeline Delay**: Owning the car and financing it from your investment budget delays your retirement by **{0:.1f} years** (Scenario C), pushing your retirement age from 50 to 61.
2. **Retirement Timeline Delay**: Owning the car and financing it from your investment budget delays your retirement by **11.2 years** (Scenario C), pushing your retirement age from 50 to 61.2.\n